# BanglaNetSim Research Benchmark — v2.0.1

**Important:** This project uses entirely synthetic data. It does not contain real ISP/customer measurements from Bangladesh, and its results should not be interpreted as real-world network performance.

## Reproduce the experiment

1. Open this notebook in Google Colab.
2. Ensure the corrected dataset is available in Google Drive at `MyDrive/BanglaNetSim2/banglanetsim_v2_0_1.csv.gz`.
3. Run the pipeline cell. It audits the dataset, trains/evaluates baseline models, and saves reports and plots under `MyDrive/BanglaNetSim2/reports/overnight_v2_0_1/`.

The notebook may take time and compute to run. You do **not** need to rerun it just to publish the existing project artifacts.


In [ ]:

# ============================================================
# BANGLANETSIM v2.0.1 — RESUMABLE RESEARCH PIPELINE
# Run as one cell in Google Colab.
# Synthetic data only; no claims about real ISP measurements.
# ============================================================

import os, json, time, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore", category=UserWarning)

# -------------------- 1. PATHS -------------------------------

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/BanglaNetSim2")
DATA = BASE / "banglanetsim_v2_0_1.csv.gz"
OUT = BASE / "reports" / "overnight_v2_0_1"
OUT.mkdir(parents=True, exist_ok=True)

if not DATA.exists():
    raise FileNotFoundError(
        f"Corrected dataset not found: {DATA}\n"
        "Check the filename in your Google Drive."
    )

print("=" * 65)
print("BANGLANETSIM RESEARCH PIPELINE")
print("Started:", datetime.now().astimezone().isoformat())
print("Dataset:", DATA)
print("Output:", OUT)
print("=" * 65)

# -------------------- 2. LOAD DATA ---------------------------

print("\n[1/6] Loading dataset...")
start = time.time()
df = pd.read_csv(DATA, compression="gzip", low_memory=False)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns "
      f"in {(time.time()-start)/60:.2f} minutes.")

required = [
    "session_id", "scenario_type", "split", "region",
    "connection_type", "access_type", "route_type",
    "plan_speed_mbps", "congestion_level",
    "wifi_signal_quality", "download_mbps", "upload_mbps",
    "latency_ms", "jitter_ms", "packet_loss_pct",
    "performance_class", "is_synthetic"
]
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")

# -------------------- 3. DATA QUALITY AUDIT ------------------

print("\n[2/6] Running integrity audit...")
numeric_cols = [
    "plan_speed_mbps", "download_mbps", "upload_mbps",
    "latency_ms", "jitter_ms", "packet_loss_pct"
]

missing_cells = int(df.isna().sum().sum())
duplicate_ids = int(df["session_id"].duplicated().sum())
finite_numeric = bool(
    np.isfinite(df[numeric_cols].to_numpy(dtype=float)).all()
)
synthetic_values = df["is_synthetic"].astype(str).str.lower()
all_synthetic = bool(synthetic_values.isin(["true", "1"]).all())

expected_classes = {"Excellent", "Good", "Fair", "Poor"}
observed_classes = set(df["performance_class"].dropna().unique())
valid_classes = observed_classes.issubset(expected_classes)

expected_splits = {"train", "validation", "test"}
observed_splits = set(df["split"].dropna().unique())
valid_splits = observed_splits.issubset(expected_splits)

audit = {
    "dataset": "BanglaNetSim",
    "version": "2.0.1",
    "synthetic_only": all_synthetic,
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "missing_cells": missing_cells,
    "duplicate_session_ids": duplicate_ids,
    "numeric_values_all_finite": finite_numeric,
    "all_rows_marked_synthetic": all_synthetic,
    "valid_performance_classes": bool(valid_classes),
    "valid_split_labels": bool(valid_splits),
    "class_distribution": {
        str(k): int(v)
        for k, v in df["performance_class"].value_counts().items()
    },
    "split_distribution": {
        str(k): int(v)
        for k, v in df["split"].value_counts().items()
    },
    "scenario_distribution": {
        str(k): int(v)
        for k, v in df["scenario_type"].value_counts().items()
    },
    "numeric_ranges": {
        col: {
            "min": float(df[col].min()),
            "max": float(df[col].max()),
            "mean": float(df[col].mean())
        }
        for col in numeric_cols
    }
}

audit["audit_passed"] = bool(
    len(df) > 0
    and missing_cells == 0
    and duplicate_ids == 0
    and finite_numeric
    and all_synthetic
    and valid_classes
    and valid_splits
)

with open(OUT / "data_quality_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

print(json.dumps(audit, indent=2))

if not audit["audit_passed"]:
    raise ValueError(
        "Integrity audit failed. Inspect data_quality_audit.json "
        "before running experiments."
    )

# Save easy-to-read distributions.
for col in ["performance_class", "split", "scenario_type",
            "congestion_level", "connection_type"]:
    df[col].value_counts().rename_axis(col).reset_index(
        name="count"
    ).to_csv(OUT / f"distribution_{col}.csv", index=False)

# -------------------- 4. LEAKAGE-SAFE SETUP ------------------

print("\n[3/6] Preparing leakage-conscious experiment...")

# ONLY pre-measurement/context features.
# Excluded: download, upload, latency, jitter, packet loss.
# Also excluded: scenario/split identifiers and record metadata.
categorical_features = [
    "region", "connection_type", "access_type",
    "route_type", "congestion_level", "wifi_signal_quality"
]
numeric_features = ["plan_speed_mbps"]
features = categorical_features + numeric_features
target = "performance_class"

train = df[df["split"] == "train"]
validation = df[df["split"] == "validation"]
test = df[df["split"] == "test"]

if min(len(train), len(validation), len(test)) == 0:
    raise ValueError("Train, validation, and test must all be non-empty.")

X_train, y_train = train[features], train[target]
X_val, y_val = validation[features], validation[target]
X_test, y_test = test[features], test[target]

print("Train:", f"{len(train):,}")
print("Validation:", f"{len(validation):,}")
print("Test:", f"{len(test):,}")
print("Features:", features)
print("Target:", target)
print("Held-out test scenarios:",
      sorted(test["scenario_type"].astype(str).unique()))

# Compatibility with older/newer scikit-learn versions.
try:
    encoder = OneHotEncoder(handle_unknown="ignore",
                            sparse_output=True)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder)
    ]), categorical_features),
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_features)
])

# -------------------- 5. CHECKPOINTED MODEL EXPERIMENTS ------

print("\n[4/6] Running models. Completed model checkpoints will be reused.")

models = {
    "dummy_baseline": DummyClassifier(strategy="most_frequent"),

    "decision_tree": DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42
    ),

    "logistic_regression": LogisticRegression(
        max_iter=150,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=5,
        max_samples=0.8,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
}

CLASS_ORDER = ["Excellent", "Fair", "Good", "Poor"]
results = {}

def save_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

for name, estimator in models.items():
    checkpoint = OUT / f"checkpoint_{name}.json"

    if checkpoint.exists():
        with open(checkpoint) as f:
            results[name] = json.load(f)
        print(f"\n[REUSE] {name}: loaded saved checkpoint.")
        continue

    print(f"\n[RUN] {name}")
    t0 = time.time()

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipeline.fit(X_train, y_train)

    model_result = {
        "model": name,
        "features_used": features,
        "target": target,
        "train_rows": int(len(train)),
        "validation_rows": int(len(validation)),
        "test_rows": int(len(test)),
        "runtime_seconds": round(time.time() - t0, 2),
        "validation": {},
        "test": {}
    }

    for split_name, X_eval, y_eval in [
        ("validation", X_val, y_val),
        ("test", X_test, y_test)
    ]:
        predictions = pipeline.predict(X_eval)

        model_result[split_name] = {
            "accuracy": float(accuracy_score(y_eval, predictions)),
            "macro_f1": float(f1_score(
                y_eval, predictions, average="macro",
                labels=CLASS_ORDER, zero_division=0
            )),
            "weighted_f1": float(f1_score(
                y_eval, predictions, average="weighted",
                labels=CLASS_ORDER, zero_division=0
            )),
            "classification_report": classification_report(
                y_eval, predictions,
                labels=CLASS_ORDER,
                target_names=CLASS_ORDER,
                output_dict=True,
                zero_division=0
            ),
            "confusion_matrix": confusion_matrix(
                y_eval, predictions, labels=CLASS_ORDER
            ).tolist()
        }

    # Save a Random Forest feature-importance report.
    if name == "random_forest":
        try:
            prep = pipeline.named_steps["preprocessor"]
            fitted_model = pipeline.named_steps["model"]
            encoded_names = prep.get_feature_names_out()
            importances = fitted_model.feature_importances_

            importance_df = pd.DataFrame({
                "encoded_feature": encoded_names,
                "importance": importances
            }).sort_values("importance", ascending=False)

            importance_df.to_csv(
                OUT / "random_forest_feature_importance.csv",
                index=False
            )
            model_result["top_features"] = (
                importance_df.head(20).to_dict(orient="records")
            )
        except Exception as e:
            model_result["feature_importance_note"] = str(e)

    results[name] = model_result
    save_json(checkpoint, model_result)
    save_json(OUT / "all_model_results.json", results)

    print(
        f"  Test accuracy: {model_result['test']['accuracy']:.4f}"
        f" | Test macro-F1: {model_result['test']['macro_f1']:.4f}"
        f" | Runtime: {model_result['runtime_seconds']:.1f}s"
    )

# -------------------- 6. TABLES, PLOTS, REPORT, ZIP ----------

print("\n[5/6] Generating tables, plots, and documentation...")

summary_rows = []
for name, result in results.items():
    for split_name in ["validation", "test"]:
        metrics = result[split_name]
        summary_rows.append({
            "model": name,
            "evaluation_split": split_name,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "runtime_seconds": result.get("runtime_seconds")
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "model_comparison.csv", index=False)

# Model comparison plot: test macro-F1.
test_summary = summary[summary["evaluation_split"] == "test"]
plt.figure(figsize=(10, 5))
plt.bar(test_summary["model"], test_summary["macro_f1"])
plt.ylabel("Test macro-F1")
plt.title("BanglaNetSim v2.0.1 — Context-Only Model Comparison")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT / "model_comparison_macro_f1.png", dpi=160)
plt.close()

# Target distribution plot.
class_counts = df["performance_class"].value_counts().reindex(
    CLASS_ORDER, fill_value=0
)
plt.figure(figsize=(8, 5))
plt.bar(class_counts.index, class_counts.values)
plt.ylabel("Number of sessions")
plt.title("Synthetic Performance-Class Distribution")
plt.tight_layout()
plt.savefig(OUT / "performance_class_distribution.png", dpi=160)
plt.close()

# Random Forest confusion matrix.
if "random_forest" in results:
    cm = np.array(results["random_forest"]["test"]["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_xticks(range(len(CLASS_ORDER)), CLASS_ORDER,
                  rotation=35, ha="right")
    ax.set_yticks(range(len(CLASS_ORDER)), CLASS_ORDER)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("Random Forest — Held-Out Test Scenarios")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}",
                    ha="center", va="center")
    fig.colorbar(im, ax=ax, label="Sessions")
    fig.tight_layout()
    fig.savefig(OUT / "random_forest_test_confusion_matrix.png",
                dpi=160)
    plt.close(fig)

# Build a clear, honest report.
rf_test = results.get("random_forest", {}).get("test", {})
best_model = (
    test_summary.sort_values("macro_f1", ascending=False).iloc[0]
    if not test_summary.empty else None
)

report_lines = [
    "# BanglaNetSim v2.0.1 — Research Report",
    "",
    f"Generated: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Dataset and scope",
    "",
    f"- Rows: {len(df):,}",
    f"- Columns: {len(df.columns)}",
    "- Data status: entirely synthetic; not real ISP measurements.",
    "- Target: simulator-generated `performance_class`.",
    "- Task: estimate the simulator-defined class from context features.",
    "",
    "## Experimental design",
    "",
    f"- Training rows: {len(train):,}",
    f"- Validation rows: {len(validation):,}",
    f"- Test rows: {len(test):,}",
    f"- Test scenarios: {', '.join(sorted(test['scenario_type'].astype(str).unique()))}",
    f"- Features: {', '.join(features)}",
    "- Excluded outcome measurements: download, upload, latency, jitter, packet loss.",
    "- Excluded metadata: scenario, split, timestamp, session ID, random seed.",
    "",
    "## Test results",
    "",
    "| Model | Accuracy | Macro-F1 | Weighted-F1 |",
    "|---|---:|---:|---:|"
]
for _, row in test_summary.iterrows():
    report_lines.append(
        f"| {row['model']} | {row['accuracy']:.4f} | "
        f"{row['macro_f1']:.4f} | {row['weighted_f1']:.4f} |"
    )

report_lines += [
    "",
    "## Important interpretation",
    "",
    "The target is derived from download speed, latency, and packet loss "
    "using rules inside the simulator. Those measurements were excluded "
    "from the predictive model inputs to avoid directly reconstructing "
    "the target from its defining measurements.",
    "",
    "Because both features and labels are synthetic, these results measure "
    "how well models learn this simulator's patterns on held-out scenarios. "
    "They do not establish performance on real Bangladeshi networks or "
    "prove generalization to real ISP data.",
    "",
    "The test split contains held-out scenarios. This is a useful stress "
    "test, but results may reflect differences between generated scenarios "
    "rather than real-world domain generalization.",
    "",
    "## Files",
    "",
    "- `data_quality_audit.json`: integrity checks and distributions.",
    "- `model_comparison.csv`: validation/test metrics.",
    "- `all_model_results.json`: detailed classification reports.",
    "- `random_forest_feature_importance.csv`: encoded feature importances, if available.",
    "- PNG files: model comparison, class distribution, and confusion matrix.",
    "",
    "## Recommended next step",
    "",
    "Validate the simulator against independently collected, consented, "
    "real network measurements before making claims about actual network "
    "performance. Document generation rules and limitations prominently."
]

report_text = "\n".join(report_lines)
(OUT / "RESEARCH_REPORT.md").write_text(report_text, encoding="utf-8")

# Package deliverables without duplicating the original dataset.
package_dir = OUT / "publication_package"
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True)

files_to_package = [
    "data_quality_audit.json",
    "all_model_results.json",
    "model_comparison.csv",
    "RESEARCH_REPORT.md",
    "random_forest_feature_importance.csv",
    "model_comparison_macro_f1.png",
    "performance_class_distribution.png",
    "random_forest_test_confusion_matrix.png"
]

for filename in files_to_package:
    source = OUT / filename
    if source.exists():
        shutil.copy2(source, package_dir / filename)

# Include all generated distribution CSVs.
for source in OUT.glob("distribution_*.csv"):
    shutil.copy2(source, package_dir / source.name)

zip_base = BASE / "BanglaNetSim_v2_0_1_research_package"
zip_path = Path(str(zip_base) + ".zip")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(package_dir)
)

# -------------------- FINISHED --------------------------------

print("\n[6/6] PIPELINE FINISHED")
print("=" * 65)
print("Data audit:", OUT / "data_quality_audit.json")
print("Comparison:", OUT / "model_comparison.csv")
print("Research report:", OUT / "RESEARCH_REPORT.md")
print("ZIP package:", zip_path)
print("\nTEST RESULTS")
print(test_summary.to_string(index=False))
print("\nFinished:", datetime.now().astimezone().isoformat())
print("=" * 65)
print(
    "\nKeep this notebook and Google Drive open until the cell finishes. "
    "Colab Free may disconnect during long runs, but completed model "
    "checkpoints are saved so you can rerun the cell to resume."
)
